# Global Terrorism Dataset

Available on Kaggle at https://www.kaggle.com/datasets/START-UMD/gtd

## Sitography
Historical notes have been retrieved from these articles
- https://www.infodata.ilsole24ore.com/2015/10/09/levoluzione-del-terrorismo-nel-tempo-1970-2014/
- https://www.agensir.it/europa/2024/10/04/terrorismo-react2024-attacchi-in-diminuzione-ma-persiste-la-minaccia/
- https://www.rsis.edu.sg/ctta-newsarticle/evolving-global-geopolitics-and-terrorism-in-south-and-southeast-asia-past-present-and-future/


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default='notebook'

In [ ]:
global_terroris_df = pd.read_csv("global_terrorism.csv", encoding="latin1", low_memory=False)
global_terroris_df.head(3)

In [ ]:
global_terroris_df.rename(columns={'iyear':'Year','imonth':'Month','iday':'Day','country_txt':'Country','region_txt':'Region','attacktype1_txt':'AttackType','target1':'Target','nkill':'Killed','nwound':'Wounded','summary':'Summary','gname':'Group','targtype1_txt':'Target_type','weaptype1_txt':'Weapon_type','motive':'Motive'},inplace=True)
global_terroris_df=global_terroris_df[['Year','Month','Day','Country','Region','city','latitude','longitude','AttackType','Killed','Wounded','Target','Summary','Group','Target_type','Weapon_type','Motive']]
global_terroris_df['casualities']=global_terroris_df['Killed']+global_terroris_df['Wounded']

# Temporal Distribution of Terrorist Attacks

## Description
This analysis shows how the number of terrorist attacks has varied from 1970 to 2017. It helps to identify temporal patterns, such as sudden increases due to historical events or periods of global conflict.

- **X-axis:** Year
- **Y-axis:** Number of Attacks

## Socio-Cultural Connection
The significant increase in attacks during the 1980s can be attributed to the Cold War and the rise of state-sponsored terrorist groups. After 2001, the graph shows a drastic surge in attacks, largely due to the "War on Terror" and destabilization in the Middle East.


In [ ]:
# Grouping the data by year and counting the number of attacks per year
attacks_per_year = global_terroris_df.groupby('Year').size().reset_index(name='attacks')

# Renaming columns for clarity
attacks_per_year.columns = ['Year', 'Number of Attacks']

# Sorting the data by Year
attacks_per_year = attacks_per_year.sort_values('Year')

# Creating a bar chart to visualize the number of attacks per year
fig = px.bar(
    attacks_per_year,
    x='Year',
    y='Number of Attacks',
    title='Number of Terrorist Attacks per Year',
    labels={'Year': 'Year', 'Number of Attacks': 'Number of Attacks'},
    color='Year',
    color_continuous_scale='Inferno'
)

# Creating a line trace for the same data to show the trend
line_trace = px.line(
    attacks_per_year,
    x='Year',
    y='Number of Attacks',
    line_shape='linear',
).data[0]

# Customizing the appearance of the line trace
line_trace.update(line=dict(color='black', width=3))
line_trace.update(mode='lines+markers', marker=dict(size=8, color='black'))

# Adding the line trace to the bar chart
fig.add_trace(line_trace)

# Customizing layout and axis labels
fig.update_layout(
    template='plotly_white',
    xaxis=dict(
        title='Year',
        tickvals=attacks_per_year['Year'],
        tickangle=45,
        showticklabels=True,
        tickson="labels",
    ),
    yaxis=dict(title='Number of Attacks'),
    title_font=dict(size=18),
    title_x=0.5
)

# Display the plot
fig.show()


# Victim Distribution by Year

## Description
This chart shows the total number of victims per year, highlighting the trend of victims over time.

- **X-axis:** Year
- **Y-axis:** Total Number of Victims

## Socio-Cultural Connection
The significant increase in victims after the September 11, 2001 attacks is linked to the "War on Terror" and global destabilization, while the years before reflect periods of local conflicts and isolated terrorist attacks.


In [ ]:
# Grouping and summing the number of victims per year
victims_per_year = global_terroris_df.groupby('Year')['casualities'].sum().reset_index(name='Total Killed + Wounded')

# Sorting by year
victims_per_year = victims_per_year.sort_values('Year')

# Creating a bar chart with custom colors
fig = px.bar(
    victims_per_year,
    x='Year',
    y='Total Killed + Wounded',
    title='Total Number of killed and wounded per Year',
    labels={'Year': 'Year', 'Total Killed + Wounded': 'Total Number of Killed + Wounded'},
    color='Year',
    color_continuous_scale='Inferno'
)

# Adding a line connecting the top of each bar
line_trace = px.line(
    victims_per_year,
    x='Year',
    y='Total Killed + Wounded',
    line_shape='linear',
).data[0]

# Modifying the line (black color and bold) directly on the trace
line_trace.update(line=dict(color='black', width=3))  # Black color and thicker line

# Adding markers on the line points
line_trace.update(mode='lines+markers', marker=dict(size=8, color='black'))  # Black markers on the points

# Adding the line trace to the chart
fig.add_trace(line_trace)

# Configuring the axis and layout
fig.update_layout(
    template='plotly_white',
    xaxis=dict(
        title='Year',
        tickvals=victims_per_year['Year'],
        tickangle=45,
        showticklabels=True,
        tickson="labels",
    ),
    yaxis=dict(title='Total Number of Killed + Wounded'),
    title_font=dict(size=18),
    title_x=0.5
)

# Displaying the plot
fig.show()


# Evolution of Attacks by Terrorist Group

## Description
This chart shows the evolution of the number of attacks by the 5 most active terrorist groups over time. The chart is divided into two sections: on the left, the lines representing the evolution of attacks for each group are visible, while on the right, the legend is separated for clarity.

- **X-axis:** Year
- **Y-axis:** Number of Attacks

## Socio-Cultural Connection
- **Groups with increasing activity:** Some groups show a significant rise in attacks in certain years, possibly indicating a strengthening of their operations.
- **Groups with constant activity:** Other groups maintain a steady frequency over time, suggesting that they may be continuously active in certain regions or specific situations.


In [ ]:
# Filter the top 10 most active terrorist groups
top_groups5 = global_terroris_df[global_terroris_df['Group'].isin(global_terroris_df['Group'].value_counts()[1:6].index)]

# Group by year and group, calculating the number of attacks for each group each year
group_yearly_attacks = pd.crosstab(top_groups5['Year'], top_groups5['Group'])

# Create the line plot with subplot: one for the chart and one for the legend
fig = make_subplots(
    rows=1, cols=2, 
    column_widths=[0.8, 0.2], 
    horizontal_spacing=0.1, 
)

# Add a trace for each group (line)
for group in group_yearly_attacks.columns:
    fig.add_trace(go.Scatter(
        x=group_yearly_attacks.index,
        y=group_yearly_attacks[group],
        mode='lines+markers',
        name=group,
        line=dict(width=3),
    ), row=1, col=1)

# Customize chart layout
fig.update_layout(
    title=dict(
        text='Evolution of Attacks by Terrorist Group (Top 5)', 
        y=0.95,
        x=0.5,
        xanchor='center',
        yanchor='top'
    ),
    xaxis_title='Year',
    yaxis_title='Number of Attacks',
    height=600,
    template='plotly',
    xaxis=dict(showgrid=True),
    yaxis=dict(showgrid=True), 
    showlegend=True, 
    legend_title=dict(
        text="Terrorist Groups", 
        font=dict(size=14)
    ),
    legend=dict(
        x=0.8, 
        bordercolor='black', 
        borderwidth=2,
        yanchor='top',
        xanchor='left',
        orientation='v' 
    ),
)

# Display the chart
fig.show()

# Types of Attack

## Description
This analysis examines the most commonly used attack methods and their impact in terms of victims. The chart will show a comparison between different types of attacks (e.g., bombings, armed assaults, etc.).
- **X-axis:** Attack type
- **Y-axis:** Number of Attacks

## Socio-Cultural Connection
Explosive attacks, common during civil wars and asymmetric conflicts, have become the deadliest method in recent decades. In contrast, armed assaults and kidnappings reflect local conflicts or political targets.


In [ ]:
# Group the data by 'AttackType' and count the number of attacks for each type
attack_counts = global_terroris_df['AttackType'].value_counts().reset_index()
attack_counts.columns = ['AttackType', 'Count']

# Create a bar chart with the inverted inferno palette
fig = px.bar(
    attack_counts,
    x='AttackType',
    y='Count',
    title='Attack Count by Type',
    labels={'AttackType': 'Attack Type', 'Count': 'Number of Attacks'},
    color='Count',
    color_continuous_scale='inferno_r'
)

# Customize the layout
fig.update_layout(
    template="plotly_white",  
    xaxis=dict(title='Attack Type', tickangle=45), 
    yaxis=dict(title='Number of Attacks'),
    height = 700
)

fig.show()


# Attack Type Trends Over Time
In this line chart, the evolution of different terrorist attack types over time is displayed, excluding the category "Unknown". The graph tracks the ranking of attack types based on the number of occurrences in 5-year intervals, highlighting how the prevalence of specific methods has changed across the decades.

## Analysis Details:
- **X-Axis**: Represents 5-year intervals, allowing us to observe trends over time.
- **Y-Axis**: Indicates the ranking of attack types based on their frequency in each time period (Rank 1 is the most frequent).
- **Lines**: Each line represents an attack type, showing how its ranking fluctuates over the years.
Exclusion: The category "Unknown" has been excluded to focus on meaningful attack classifications.
## Key Insights:
- **Most Prevalent Types**: Certain attack types, such as Bombings/Explosions or Armed Assaults, consistently rank high, reflecting their widespread use and effectiveness in causing large-scale impact.
Emerging Trends: Over time, some attack types may rise in rank, indicating emerging tactics or shifts in strategic preferences by terrorist groups.
- **Declining Methods**: Conversely, some methods might decline in rank, possibly due to increased countermeasures or shifts in global geopolitics.
## Discussion Points:
The graph offers a basis to discuss how global events (e.g., wars, technological advancements, policy changes) may influence the choice of attack methods.
The focus on rankings instead of absolute numbers helps to highlight relative changes in the prevalence of different attack types.

In [ ]:
# Group by 5-year intervals
global_terroris_df['Year_Group'] = (global_terroris_df['Year'] // 5) * 5

# Filter out rows where AttackType is "Unknown"
filtered_df = global_terroris_df[global_terroris_df['AttackType'] != "Unknown"]

# Calculate the number of terrorist attacks per attack type and 5-year interval
agg_data = (
    filtered_df
    .groupby(['Year_Group', 'AttackType'])
    .size()
    .reset_index(name='AttackCount')
)

# Identify the top 10 attack types in the most recent time period
latest_year = agg_data['Year_Group'].max()
top_attack_types = (
    agg_data[agg_data['Year_Group'] == latest_year]
    .nlargest(10, 'AttackCount')['AttackType']
    .tolist()
)

# Filter for the top 10 attack types
agg_data = agg_data[agg_data['AttackType'].isin(top_attack_types)]

# Ranking the attack types for each 5-year period
agg_data['Rank'] = (
    agg_data.groupby('Year_Group')['AttackCount']
    .rank(method="dense", ascending=False)
)

# Create the time series plot with ranking on the y-axis
fig = px.line(
    agg_data,
    x="Year_Group",
    y="Rank",
    color="AttackType",
    title="Terrorist Attack Ranking by Attack Type Over Time",
    labels={
        "AttackType": "Attack Type"
    },
    markers=True,
    color_discrete_sequence=px.colors.qualitative.Pastel,
    line_shape='linear',
)

# Customize the layout
fig.update_layout(
    yaxis=dict(
        tickmode="linear",
        dtick=1,
        autorange="reversed",
        title=None
    ),
    xaxis=dict(
        tickmode="linear",
        dtick=5,
        title=None
    ),
    legend_title="Attack Type"
)

fig.update_traces(line=dict(width=6))
fig.show()


# Total Killed + Wounded by Attack Type

## Description
This analysis explores the impact of various attack types based on the total number of casualties, including both those killed and wounded. The chart highlights the most lethal methods of attack, offering insights into their overall toll.  
- **X-axis:** Attack Type  
- **Y-axis:** Total Number of Killed + Wounded  

## Socio-Cultural Connection
Certain attack methods, such as bombings and armed assaults, account for the majority of casualties worldwide. These methods are often used to cause maximum harm in high-population areas or critical infrastructure. Other tactics, such as assassinations or facility attacks, tend to target specific individuals or organizations, showing the diverse strategies in terrorist operations.


In [ ]:
# Group by attack type and calculate total casualties
attack_casualties = global_terroris_df.groupby('AttackType')['casualities'].sum().reset_index()

# Sort by total casualties in descending order
attack_casualties = attack_casualties.sort_values(by='casualities', ascending=False)

# Create a bar chart for total casualties by attack type
fig = px.bar(
    attack_casualties,
    x='AttackType',
    y='casualities',
    title='Total Killed + Wounded by Attack Type',
    labels={'AttackType': 'Attack Type', 'casualities': 'Total Killed + Wounded'},
    color='casualities',
    color_continuous_scale='Inferno_r'
)

# Rotate x-axis labels for better readability
fig.update_layout(
    xaxis_tickangle=45,
    xaxis_title="Attack Type",
    yaxis_title="Total Killed + Wounded",
    height=700
)

# Show the plot
fig.show()


# Target Type and Attack Frequency Analysis

In this bar chart, we analyze the distribution of terrorist attacks based on **target types**. Each bar represents a target type (e.g., government buildings, civilians, security forces, etc.), while the height of the bar indicates the number of attacks recorded for that target type.

## Analysis Details:
- **X-Axis**: Represents the different **target types**, as identified in the `Target_type` column of the dataset.
- **Y-Axis**: Shows the **number of attacks** carried out on each target type.

## Socio-Cultural Connection:
This chart offers an interesting insight into the priorities of terrorist attacks over the years, based on target types. For example, we can observe a significant concentration of attacks against **security forces** or **government institutions**, often linked to the struggle against state power, police forces, or a country's militarization. Additionally, historical events such as wars or armed conflicts can increase the frequency of attacks against specific targets, such as **military** forces or **political opponents**.


In [ ]:
# Group the data by 'Target_type' and count the number of attacks for each target type
target_counts = global_terroris_df['Target_type'].value_counts().reset_index()
target_counts.columns = ['Target_type', 'Count']

# Sort the values in descending order
target_counts = target_counts.sort_values(by='Count', ascending=False)

# Create a bar chart with the inverted inferno palette
fig = px.bar(
    target_counts,
    x='Target_type',
    y='Count',
    title='Attack Count by Target Type',
    labels={'Target_type': 'Target Type', 'Count': 'Number of Attacks'},
    color='Count',
    color_continuous_scale='inferno_r', 
)

# Customize the layout
fig.update_layout(
    template="plotly_white",
    xaxis=dict(title='Target Type', tickangle=45),
    yaxis=dict(title='Number of Attacks'),
    height = 700
)

fig.show()


# Number of Attacks by Region Analysis

In this bar chart, the number of terrorist attacks for each region of the world is displayed using data from the Global Terrorism Database (GTD). The X-axis represents the geographical regions, while the Y-axis indicates the total number of attacks that occurred in each of these regions.

## Analysis Details:
- **X-Axis**: Represents the different **regions** around the world, as identified in the `Region` column of the dataset.
- **Y-Axis**: Shows the **total number of attacks** that occurred in each region.

## Socio-Cultural Connection:
- **Some regions stand out with significantly higher** attack numbers, such as the **Middle East** and **Africa**. These areas have historically and geopolitically been vulnerable to armed conflicts, political instability, and terrorism due to a combination of factors, including civil wars, poverty, and government instability.
  
- **Other regions** show fewer attacks, suggesting a less active presence of terrorist groups or greater political stability and security. However, it is important to note that a lower number of attacks does not automatically imply a lack of terrorist risk but may also reflect the difficulty in gathering and recording data from certain areas.


In [ ]:
# Group the data by region and count the number of attacks for each region
region_counts = global_terroris_df['Region'].value_counts().reset_index()
region_counts.columns = ['Region', 'Attack_Count']

# Create the bar chart
fig = px.bar(region_counts, 
             x='Region', 
             y='Attack_Count', 
             color='Attack_Count', 
             color_continuous_scale='inferno_r', 
             title="Number of Attacks by Region", 
             labels={'Attack_Count': 'Number of Attacks', 'Region': 'Region'})

# Sort the bars in descending order based on the number of attacks
fig.update_layout(
    xaxis={'categoryorder': 'total descending'},
    height = 700
)

# Show the chart
fig.show()


# Terrorist Attack Ranking by Region Over Time

In this line chart, the ranking of terrorist attacks by region over a period of 5 years is presented, using data from the Global Terrorism Database (GTD). The chart shows how the regions ranked in terms of the number of attacks during each 5-year interval.

## Analysis Details:
- **X-Axis**: Represents the 5-year intervals (Year Group), which allows us to track the number of terrorist attacks over time, grouped into five-year periods.
- **Y-Axis**: Displays the ranking of each region for every 5-year period, with Rank 1 indicating the region with the highest number of attacks.
- **Color**: Different colors represent different regions, making it easy to distinguish between the regions across the time periods.

## Key Insights:
- The **Middle East** and **Africa** often rank at the top of the chart, reflecting their historical challenges with terrorism due to a combination of political instability, ongoing conflicts, and regional power struggles.
- Some **European** regions have a lower ranking, which could suggest greater stability and more effective counter-terrorism measures. However, even in these regions, occasional spikes in terrorist activities can occur, requiring continuous vigilance and adaptation of security policies.
- The **rankings over time** provide insight into the fluctuations in terrorist activities in different regions. For instance, there may be a noticeable rise in certain years for regions that are experiencing political turmoil or military interventions.

## Socio-Cultural Connection:
Regions with frequent attacks often face a mix of socio-political challenges, such as armed conflicts, failed states, or terrorist group activity. The global fight against terrorism involves addressing root causes like poverty, lack of governance, and regional conflicts, while balancing security and human rights concerns. Understanding these trends helps in planning more targeted interventions, improving regional security, and fostering international cooperation to combat terrorism.

This visualization serves as a reminder of the ongoing global challenge posed by terrorism and the importance of understanding regional dynamics to effectively address it.

In [ ]:
# Group by 5-year intervals
global_terroris_df['Year_Group'] = (global_terroris_df['Year'] // 5) * 5

# Calculate the number of terrorist attacks per region and 5-year interval
agg_data = (
    global_terroris_df
    .groupby(['Year_Group', 'Region'])
    .size()
    .reset_index(name='AttackCount')
)

# Identify the top 10 regions in the most recent time period
latest_year = agg_data['Year_Group'].max()
top_regions = (
    agg_data[agg_data['Year_Group'] == latest_year]
    .nlargest(10, 'AttackCount')['Region']
    .tolist()
)

# Filter for the top 10 regions
agg_data = agg_data[agg_data['Region'].isin(top_regions)]

# Ranking the regions for each 5-year period
agg_data['Rank'] = (
    agg_data.groupby('Year_Group')['AttackCount']
    .rank(method="dense", ascending=False)
)

# Create the time series plot with ranking on the y-axis
fig = px.line(
    agg_data,
    x="Year_Group",
    y="Rank",
    color="Region",
    title="Terrorist Attack Ranking by Region Over Time",
    labels={
        "Region": "Region"
    },
    markers=True,
    color_discrete_sequence=px.colors.qualitative.Pastel,
    line_shape='linear',
)

fig.update_layout(
    yaxis=dict(
        tickmode="linear",
        dtick=1,
        autorange="reversed",
        title="Ranking"
    ),
    xaxis=dict(
        tickmode="linear",
        dtick=5,
        title="Year"
    ),
    legend_title="Region"
)

fig.update_traces(line=dict(width=6))
fig.show()


# Success Rate of Terrorist Attacks

In this analysis, we calculated the **success rate** of terrorist attacks for each country, considering an attack to be "successful" if it caused at least one victim, either dead or wounded. The success rate is calculated as the percentage of attacks with victims compared to the total number of attacks in each country.

## Analysis Details:
- **X-axis**: Represents the **countries or regions**, as identified in the `Region` column of the dataset.
- **Y-axis**: Displays the **success rate** of attacks, calculated as the percentage of attacks that caused at least one victim compared to the total number of attacks in each country or region.

## Objective:
- To verify if more developed countries, with advanced security infrastructures, have a lower success rate for terrorist attacks. The idea is that countries with better resources might have a lower chance of an attack being successful, meaning causing victims, due to preventive and response measures.

## Interpretation:
- Countries with a **low success rate** of attacks could be those with **higher prevention capabilities** and **security**.
- Countries with a **high success rate** could be facing **political instability** or have **less developed security**.

## Result:
In the shown graph, countries with a high success rate are those that have recorded more lethal or severe attacks. On the other hand, countries with a lower success rate may have implemented stronger security measures, reducing the effectiveness of terrorist attacks.
This graph provides insight into the effectiveness of security policies in various countries and how they might influence the success of terrorist attacks.


In [ ]:
# Create a new column indicating whether the attack was "successful" or not
global_terroris_df['Success'] = global_terroris_df['Killed'].fillna(0) + global_terroris_df['Wounded'].fillna(0) > 0

# Calculate the success rate by country (or region)
country_success_rate = global_terroris_df.groupby('Region')['Success'].mean() * 100

# Sort countries by success rate in descending order
country_success_rate = country_success_rate.sort_values(ascending=False)

# Create the bar chart
fig = px.bar(country_success_rate,
             x=country_success_rate.index,
             y=country_success_rate,
             labels={'x': 'Region', 'y': 'Success Rate (%)'},
             title='Success Rate of Terrorist Attacks by Country',
             color=country_success_rate,
             color_continuous_scale='Inferno_r') 

# Customize the appearance
fig.update_layout(
    xaxis_title="Region",
    yaxis_title="Success Rate (%)",
    height=700
)

# Show the chart
fig.show()


# Attacks and Victims in the Top 15 Countries with Double Bars

This bar chart shows the top 15 countries with the highest number of terrorist attacks (attacks) and the total number of victims (killed), organized with double bars for each country. The red bars represent the number of attacks, while the blue bars represent the total number of victims.

## Data Encoding:
- **Attacks**: The count of terrorist attacks for each country (represented by red bars).
- **Victims (Killed)**: The total number of victims (killed) for each country (represented by blue bars).

## Objective:
- To examine the relationship between the number of terrorist attacks and the number of victims in the top 15 most affected countries. The red and blue bars allow for a comparison between the intensity of the attacks and their human impact in terms of victims.

## Interpretation:
- **Countries with the Most Attacks**: Some of the countries with the highest number of attacks are those that have experienced prolonged conflicts or instability. These attacks have significantly impacted the number of victims.

## Conclusion:
This visualization provides a clear overview of the relationship between the number of terrorist attacks and the number of victims in the top 15 affected countries. The double bars make it easy to compare the intensity of attacks with their human impact in terms of victims.


In [ ]:
# Group the data by country and calculate the number of attacks and the total number of victims
coun_terror = global_terroris_df['Country'].value_counts()[:15].reset_index()
coun_terror.columns = ['Country', 'Attacks']

coun_kill = global_terroris_df.groupby('Country')['Killed'].sum().reset_index()
coun_kill.columns = ['Country', 'Killed']

# Merge the two DataFrames
data = pd.merge(coun_terror, coun_kill, on='Country', how='left')

# Create the bar chart
fig = go.Figure()

# Add bars for the number of attacks (red)
fig.add_trace(go.Bar(
    x=data['Country'],
    y=data['Attacks'],
    name='Attacks',
    marker=dict(color='rgb(255, 99, 71)'),
    width=0.4,
    offsetgroup=0
))

# Add bars for the number of victims (blue)
fig.add_trace(go.Bar(
    x=data['Country'],
    y=data['Killed'],
    name='Killed',
    marker=dict(color='rgb(0, 0, 255)'),
    width=0.4,
    offsetgroup=1
))

# Customize the layout of the graph
fig.update_layout(
    title='Top 15 Countries with Most Attacks and Victims',
    xaxis_title='Countries',
    yaxis_title='Number of Attacks / Victims',
    barmode='group',
    xaxis={'categoryorder': 'total descending'},
    height=600,
    template='plotly',
    legend_title="Data Type",
    legend=dict(x=0.8, y=1),
)

# Show the graph
fig.show()


# Evolution of Terrorist Attacks Over Time

This animated map visualizes the evolution of terrorist attacks over time. Each point represents an attack, with its size proportional to the number of casualties and its color indicating the severity of the attack. The color scale ranges from blue (no casualties) to red (high casualties). The animation plays over the years, showing the geographical distribution of attacks and their intensity as time progresses.

## Key Features:
- **Proportional size**: The size of each point corresponds to the number of casualties in the attack.
- **Color scale**: The points are colored from blue (no casualties) to red (high casualties).
- **Animation**: The map animates over time, showing how terrorist attacks evolve across different regions and years.
- **Interactive features**: You can hover over the points to see more details about the attack, including the year and number of casualties.

This visualization helps in understanding the global impact of terrorist attacks and how they have changed over time.


In [ ]:
# Replace NaN with 0 for "casualties" and remove rows with missing values for lat/lon
df_geo = global_terroris_df.fillna({"casualities": 0}).dropna(subset=["latitude", "longitude"])

# Normalize the casualties values to control their size, assigning a minimum value for 0
df_geo["Normalized_Casualties"] = (
    df_geo["casualities"] / df_geo["casualities"].max() * 100
).fillna(0)
df_geo.loc[df_geo["Normalized_Casualties"] == 0, "Normalized_Casualties"] = 5  # Minimum size for the points

# Convert Year to integers (if necessary)
df_geo["Year"] = df_geo["Year"].astype(int)

# Create a 'Color' column that assigns blue for 0 and a color scale for values greater than 0
df_geo["Color"] = df_geo["casualities"].apply(lambda x: 0 if x == 0 else x)  # 0 -> blue, others -> scale

# Create a scatter_mapbox plot
fig = px.scatter_mapbox(
    df_geo,
    lat="latitude",
    lon="longitude",
    size="Normalized_Casualties",
    color="Color",
    hover_name="city",
    hover_data={"Year": True, "casualities": True},
    animation_frame="Year",
    color_continuous_scale=["blue", "red"],
    title="Evolution of Terrorist Attacks Over Time"
)

# Update the map style
fig.update_layout(
    mapbox_style="carto-positron",
    mapbox_zoom=1.5,
    mapbox_center={"lat": 0, "lon": 0},
    height=800,
    margin={"r": 0, "t": 50, "l": 0, "b": 0},
    coloraxis_colorbar=dict(
        title="Killed + Wounded",
        ticks="outside",
    ),
    updatemenus=[  # Adds buttons for animation control
        {
            "buttons": [
                {
                    "args": [None, {"frame": {"duration": 1500, "redraw": True}, "fromcurrent": True}],
                    "label": "Play",
                    "method": "animate",
                },
                {
                    "args": [[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate", "transition": {"duration": 0}}],
                    "label": "Pause",
                    "method": "animate",
                },
            ],
            "direction": "left",
            "pad": {"r": 10, "t": 87},
            "showactive": False,
            "type": "buttons",
            "x": 0.1,
            "xanchor": "right",
            "y": 0,
            "yanchor": "top",
        }
    ],
)

# Display the plot
fig.show()


# Top 5 Terrorist Groups by Number of Attacks

## Key Features:
- **Top Groups**: Only the top 5 groups with the most attacks are displayed, excluding incidents attributed to "Unknown."
- **Proportional Marker Size**: The size of each point reflects the normalized number of casualties for the attack, ensuring visibility across varying scales.
- **Color-Coded Groups**: Each group is represented by a distinct color for easy identification.
- **Interactive Filtering**: Use the legend to toggle the visibility of specific groups and focus on their activity patterns.
- **Geographical Scope**: The map provides a global overview, zoomed out to show all affected regions.

## Subtitle: A Socio-Cultural Perspective on Global Terrorism
Terrorism is often deeply rooted in socio-cultural, political, and historical contexts. This visualization sheds light on the global activity patterns of the top 5 most active terrorist groups, offering insights into their geographical presence and potential socio-political connections.

This interactive map highlights the locations of terrorist attacks carried out by the top 5 most active terrorist groups worldwide (excluding "Unknown"). Each point on the map represents an attack, with its size proportional to the number of casualties (killed + wounded) and its color corresponding to the responsible group.

In [ ]:
# Prepare the dataset
df = global_terroris_df.copy()

# Exclude "Unknown" group and get the top 5 groups with the most attacks
top_groups = df['Group'].value_counts().drop('Unknown', errors='ignore').head(5).index
df_top_groups = df[df['Group'].isin(top_groups)].dropna(subset=["latitude", "longitude"])

# Assign unique colors to each group
color_map = {group: px.colors.qualitative.Plotly[i] for i, group in enumerate(top_groups)}
df_top_groups['Color'] = df_top_groups['Group'].map(color_map)

# Normalize casualties for marker size
df_top_groups['Normalized_Casualties'] = (
    df_top_groups['casualities'] / df_top_groups['casualities'].max() * 50
).fillna(5)  # Minimum marker size

# Create the map plot
map_plot = px.scatter_mapbox(
    df_top_groups,
    lat="latitude",
    lon="longitude",
    size="Normalized_Casualties",
    color="Group",
    hover_name="city",
    hover_data={
        "Group": True,
        "casualities": True,
        "Country": True,
        "Year": True,
    },
    color_discrete_map=color_map,
    title="Top 5 Terrorist Groups by Number of Attacks",
)

# Update map style and layout
map_plot.update_layout(
    mapbox_style="carto-positron", 
    mapbox_zoom=1.2, 
    mapbox_center={"lat": 0, "lon": 0}, 
    height=800,
    margin={"r": 0, "t": 40, "l": 0, "b": 0},
    legend_title="Terrorist Groups",
)

# Hide axis and disable continent names
map_plot.update_layout(
    xaxis_visible=False,
    yaxis_visible=False,
    mapbox_layers=[
        {
            "source": "",
            "sourcetype": "vector",
            "type": "line",
            "below": "traces",
        }
    ],
)

# Show the plot
map_plot.show()
